In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

backend = BasicSimulator()

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

## Quantum Random Number Generator

In [ ]:
def quantum_random_bits(n):
    """Generates n random bits using quantum measurement of the |+> state."""
    bits = []
    for _ in range(n):
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.measure(0, 0)
        t_qc = transpile(qc, backend)
        result = backend.run(t_qc, shots=1).result()
        bits.append(int(list(result.get_counts().keys())[0]))
    return bits

## Alice's Part

In [ ]:
NUM_QUBITS = 100
alice_bits = quantum_random_bits(NUM_QUBITS)
alice_bases = quantum_random_bits(NUM_QUBITS)

alice_qubits = []
for i in range(NUM_QUBITS):
    qc = QuantumCircuit(1)
    if alice_bits[i] == 1: qc.x(0)
    if alice_bases[i] == 1: qc.h(0)
    alice_qubits.append(qc)

## Eve's Attack (Intercept-Resend)

In [ ]:
eve_bases = quantum_random_bits(NUM_QUBITS)
intercepted_qubits = []

for i in range(NUM_QUBITS):
    qc = alice_qubits[i].copy()
    if eve_bases[i] == 1: qc.h(0)
    qc.add_register(__import__('qiskit').ClassicalRegister(1))
    qc.measure(0, 0)
    t_qc = transpile(qc, backend)
    measured_bit = int(list(backend.run(t_qc, shots=1).result().get_counts().keys())[0])
    
    new_qc = QuantumCircuit(1)
    if measured_bit == 1: new_qc.x(0)
    if eve_bases[i] == 1: new_qc.h(0)
    intercepted_qubits.append(new_qc)

## Bob's Part

In [ ]:
bob_bases = quantum_random_bits(NUM_QUBITS)
bob_bits = []
for i in range(NUM_QUBITS):
    qc = intercepted_qubits[i].copy()
    if bob_bases[i] == 1: qc.h(0)
    qc.add_register(__import__('qiskit').ClassicalRegister(1))
    qc.measure(0, 0)
    t_qc = transpile(qc, backend)
    bob_bits.append(int(list(backend.run(t_qc, shots=1).result().get_counts().keys())[0]))

## Detection

In [ ]:
matching = [i for i in range(NUM_QUBITS) if alice_bases[i] == bob_bases[i]]
alice_key = [alice_bits[i] for i in matching]
bob_key = [bob_bits[i] for i in matching]

check_len = len(alice_key) // 2
errors = sum(1 for i in range(check_len) if alice_key[i] != bob_key[i])
error_rate = errors / check_len if check_len > 0 else 0

print(f"Error rate: {error_rate:.2%}")
if error_rate > 0.12:
    print("ATTACK DETECTED!")